# Day 09 — Train raw vs clean trên Kaggle GPU

Notebook thay thế Colab khi không có GPU. Trong Settings: chọn GPU T4/P100 và bật Internet. Nếu tài khoản yêu cầu xác minh, chủ tài khoản tự hoàn tất. Chạy lần lượt các ô. Giữ nguyên 2 seed, 20 epoch, cùng model/val/test; clean là bản nháp Hold, không phải dataset đã Release.


In [ ]:
REPO_URL = 'https://github.com/chuns29/K5-L23-Day09-DuongQuangChung-2A202603217-AutoQC-Pipeline-Lab.git'
REPO_COMMIT = 'a0d8e07fe7de55ed9d2336457a398a6d20cb6693'
SEEDS = [0, 1]
EPOCHS = 20
VARIANTS = ["raw", "clean"]


In [ ]:
from pathlib import Path
import subprocess, sys, os, json, hashlib
import torch
assert torch.cuda.is_available(), "Chưa có GPU. Chọn Accelerator GPU trong Kaggle Settings rồi chạy lại."
subprocess.run(["nvidia-smi", "-L"], check=True)
repo_dir = Path("/kaggle/working/day09_autoqc_lab")
if not repo_dir.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)
os.chdir(repo_dir)
subprocess.run(["git", "checkout", "--detach", REPO_COMMIT], check=True)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip() == REPO_COMMIT
subprocess.run([sys.executable, "-m", "pip", "install", "ultralytics==8.4.214", "imagehash", "pyyaml", "matplotlib"], check=True)


In [ ]:
manifest_path = Path("submission/export/clean_v1/manifest.json")
manifest = json.loads(manifest_path.read_text())
assert manifest["n_images"] == 205
print("GPU:", torch.cuda.get_device_name(0))
print("Input commit:", REPO_COMMIT)
print("Dataset decision:", manifest["decision"]["decision"])
print("Clean images:", manifest["n_images"])
print("Manifest SHA1:", hashlib.sha1(manifest_path.read_bytes()).hexdigest())


In [ ]:
# 4 lượt train: raw seed 0/1 và clean seed 0/1; kết quả do script đề sinh.
subprocess.run([sys.executable, "scripts/train_compare.py", "--variants", *VARIANTS,
                "--seeds", *map(str, SEEDS), "--epochs", str(EPOCHS), "--device", "0",
                "--skip-existing"], check=True)


In [ ]:
subprocess.run([sys.executable, "scripts/summarize_results.py"], check=True)
from IPython.display import display, Markdown, Image
print(Path("submission/results/comparison.md").read_text())
display(Image("submission/results/comparison.png"))
results = [json.loads(p.read_text()) for p in sorted(Path("submission/results").glob("*_seed*.json"))]
assert {(r["variant"], r["seed"]) for r in results} == {(v,s) for v in VARIANTS for s in SEEDS}
assert all(r["epochs"] == EPOCHS for r in results)


In [ ]:
# Bằng chứng số ảnh thực sự được loader chấp nhận và môi trường train.
import numpy as np, platform, ultralytics
metadata = {"platform": "Kaggle", "input_commit": REPO_COMMIT,
            "python": platform.python_version(), "torch": torch.__version__,
            "cuda": torch.version.cuda, "ultralytics": ultralytics.__version__,
            "gpu": torch.cuda.get_device_name(0), "datasets": {}}
for variant in VARIANTS:
    cache = np.load(f"work/train/{variant}/train/labels.cache", allow_pickle=True).item()
    metadata["datasets"][variant] = {"eligible_images": len(cache["labels"]),
        "scan_results": [int(v) for v in cache["results"]], "warnings": list(cache["msgs"])}
Path("submission/results/training_environment.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False)+"\n")
print(json.dumps(metadata, ensure_ascii=False, indent=2))

# Tải results.zip từ phần Output của Kaggle, giữ nguyên JSON do script sinh.
import zipfile
archive = Path("/kaggle/working/results.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(Path("submission/results").glob("*")):
        if p.is_file() and p.name != ".gitkeep":
            z.write(p, p.as_posix())
from IPython.display import FileLink
display(FileLink(str(archive)))
print("Archive:", archive)
